# General Health Query Chatbot

## Objective
The objective of this project is to build a chatbot that answers general health-related questions using a Large Language Model (LLM). The chatbot uses prompt engineering to generate helpful responses and includes safety filters to avoid potentially harmful medical advice.

## Install Required Libraries

In this step, we install the required libraries needed to load and interact with the language model.

In [ ]:
!pip install -U transformers sentencepiece accelerate

## Import Libraries

The required libraries are imported to access Hugging Face models and build the chatbot.

In [ ]:
from transformers import pipeline
print("Pipeline imported successfully")

Pipeline imported successfully


## Load the Language Model

The TinyLlama model is loaded using the Hugging Face pipeline. This model will be responsible for generating responses to user health queries.

### Model Selection

TinyLlama-1.1B-Chat-v1.0 was selected because it is a lightweight open-source Large Language Model suitable for conversational applications and can be used without paid API access. It provides a practical solution for building a health query chatbot while meeting the project requirements of using an LLM.

In [ ]:
chatbot = pipeline("text-generation",model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
print("Model loaded successfully!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded successfully!


## Implement Safety Filter

A safety filter is added to identify potentially dangerous health situations. Instead of providing medical advice in emergency situations, the chatbot recommends seeking professional medical assistance.

In [ ]:
def safety_filter(user_query):
    """
    Detect potentially dangerous medical situations.
    """
    dangerous_keywords = [
        "suicide",
        "overdose",
        "heart attack",
        "chest pain",
        "cannot breathe",
        "can't breathe",
        "emergency",
        "severe bleeding",
        "unconscious"
    ]
    query_lower = user_query.lower()
    for keyword in dangerous_keywords:
        if keyword in query_lower:
            return (
                "This may be a medical emergency. "
                "Please seek immediate medical attention "
                "or contact emergency services."
            )
    return None

## Prompt Engineering

Prompt engineering is used to guide the model's behavior. The prompt instructs the model to provide general health information, avoid diagnoses, and maintain a friendly tone.

In [ ]:
SYSTEM_PROMPT = """
You are a helpful medical assistant.

Rules:
- Provide general health information only.
- Do not diagnose diseases.
- Do not prescribe medication.
- Use simple and friendly language.
- Keep answers concise.
- Encourage consulting a healthcare professional when necessary.
- Limit responses to 3-4 sentences.
"""

## Generate Responses

This function combines the safety filter and prompt template to generate safe and relevant responses.

In [ ]:
def get_health_response(user_query):
    """
    Generate a health-related response.
    """
    safety_response = safety_filter(user_query)
    if safety_response:
        return safety_response

    # Create prompt
    prompt = f"""
    {SYSTEM_PROMPT}

    User Question:
    {user_query}

    Assistant Response:
    """

    # Generate answer
    response = chatbot(
        prompt,
        max_new_tokens=80,
        temperature=0.3,
        do_sample=False
    )

    generated_text = response[0]["generated_text"]

    # Extract response portion
    if "Assistant Response:" in generated_text:
        generated_text = generated_text.split("Assistant Response:")[-1]

    return generated_text.strip()

## Interactive Chatbot

The chatbot allows users to ask multiple health-related questions until they choose to exit the conversation.

In [ ]:
print("===================================")
print(" General Health Query Chatbot")
print(" Type 'exit' to quit")
print("===================================\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":

        print("\nChatbot: Goodbye! Stay healthy.")
        break

    answer = get_health_response(user_input)

    print("\nChatbot:", answer)
    print()

 General Health Query Chatbot
 Type 'exit' to quit

You: what cause a sore throat?


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot: Sore throat is caused by a viral infection or bacterial infection. Some common causes include:

- Cold or flu
- Strep throat
- Laryngitis
- Voice box infection

To prevent the spread of the virus, avoid touching your mouth, nose, or eyes. Wash your hands frequently with soap and water

You: Is paracetamol safe for children?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot: Yes, paracetamol is generally safe for children. It is a non-steroidal anti-inflammatory drug (NSAID) that is used to relieve pain and inflammation. Paracetamol is not addictive and does not cause dependence. It is recommended for children under 12 years of age. However, it is

You: How can I prevent the flu?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot: Flu prevention tips:
    
    1. Get a flu shot: The flu shot is the best way to prevent the flu. It contains a combination of vaccines that protect against several strains of the flu virus.
    
    2. Wash your hands: Clean your hands often with soap and water or use an alcohol-based hand sanit

You: exit

Chatbot: Goodbye! Stay healthy.


## Testing the Chatbot

The chatbot is tested using sample health-related questions to evaluate its responses and verify that the safety filter works correctly.

In [ ]:
test_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "How can I prevent the flu?",
    "What are common symptoms of dehydration?",
    "I have chest pain and cannot breathe"
]

for question in test_questions:

    print("Question:", question)
    print("Answer:", get_health_response(question))
    print("-" * 80)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What causes a sore throat?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: A sore throat is caused by a viral infection or bacterial infection. The virus can cause a sore throat, while the bacteria can cause a bacterial infection. The virus can be spread through coughing, sneezing, or close contact with an infected person. The bacteria can be spread through close contact
--------------------------------------------------------------------------------
Question: Is paracetamol safe for children?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Yes, paracetamol is generally safe for children. It is a non-steroidal anti-inflammatory drug (NSAID) that is used to relieve pain and inflammation. Paracetamol is not addictive and does not cause dependence. It is recommended for children under 12 years of age. However, it is
--------------------------------------------------------------------------------
Question: How can I prevent the flu?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Flu prevention tips:
    
    1. Get a flu shot: The flu shot is the best way to prevent the flu. It contains a combination of vaccines that protect against several strains of the flu virus.
    
    2. Wash your hands: Clean your hands often with soap and water or use an alcohol-based hand sanit
--------------------------------------------------------------------------------
Question: What are common symptoms of dehydration?
Answer: Common symptoms of dehydration include:
    
    1. Dry mouth
    2. Thirst
    3. Difficulty in swallowing
    4. Fatigue
    5. Dry skin
    6. Blurred vision
    7. Dizziness
    8. Headache
    9. N
--------------------------------------------------------------------------------
Question: I have chest pain and cannot breathe
Answer: ⚠️ This may be a medical emergency. Please seek immediate medical attention or contact emergency services.
--------------------------------------------------------------------------------


## Results and Insights

The chatbot successfully answered general health-related questions using a Large Language Model.

Key Findings:
- Prompt engineering improved response quality and consistency.
- The safety filter prevented potentially harmful responses.
- The chatbot provided user-friendly explanations for common health queries.
- The model is suitable for educational and informational purposes but should not replace professional medical advice.

## Conclusion

A health query chatbot was successfully developed using the TinyLlama Large Language Model. Prompt engineering improved response quality, while safety filters helped prevent potentially harmful medical guidance. The chatbot can answer general health questions but should not replace professional healthcare advice.